In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

In [ ]:
df = pd.read_csv(r"C:\Users\Sruthi K\OneDrive\Desktop\Customer-Churn-Analytics\data\telecom_churn.csv")
df.head()

In [ ]:
df.shape

In [ ]:
df.columns.tolist()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

In [ ]:
df["TotalCharges"].head(10)

In [ ]:
(df["TotalCharges"] == " ").sum()

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.shape

In [ ]:
import sys
print(sys.executable)

In [ ]:
import plotly
import nbformat

print(plotly.__version__)
print(nbformat.__version__)

In [ ]:
# Drop customerID - not useful for modeling/EDA (safe to re-run)
if 'customerID' in df.columns:
    df.drop('customerID', axis=1, inplace=True)

# Convert SeniorCitizen to Yes/No for readability in plots (only if still 0/1)
if df['SeniorCitizen'].dtype in ['int64', 'float64']:
    df['SeniorCitizen'] = df['SeniorCitizen'].map({0: 'No', 1: 'Yes'})

In [ ]:
# --- Churn distribution ---
churn_counts = df['Churn'].value_counts()
churn_rate = (churn_counts['Yes'] / len(df)) * 100
print(f"Churn Rate: {churn_rate:.2f}%")

fig = px.pie(df, names='Churn', title='Customer Churn Distribution', color='Churn', color_discrete_map={'Yes':'#EF553B','No':'#00CC96'})
fig.show()

In [ ]:
# --- Churn by Contract type ---
fig = px.histogram(df, x='Contract', color='Churn', barmode='group',
                    title='Churn by Contract Type')
fig.show()

In [ ]:
# --- Churn by tenure ---
fig = px.histogram(df, x='tenure', color='Churn', nbins=30,
                    title='Churn by Tenure', marginal='box')
fig.show()

In [ ]:
# --- Churn by Monthly Charges ---
fig = px.box(df, x='Churn', y='MonthlyCharges', color='Churn',
             title='Monthly Charges vs Churn')
fig.show()

In [ ]:
# --- Churn by Internet Service ---
fig = px.histogram(df, x='InternetService', color='Churn', barmode='group',
                    title='Churn by Internet Service')
fig.show()

In [ ]:
# --- Churn by Payment Method ---
fig = px.histogram(df, x='PaymentMethod', color='Churn', barmode='group',
                    title='Churn by Payment Method')
fig.show()

In [ ]:
# --- Correlation heatmap for numeric features ---
df_corr = df.copy()
df_corr['Churn_num'] = df_corr['Churn'].map({'Yes':1,'No':0})
numeric_cols = ['tenure','MonthlyCharges','TotalCharges','Churn_num']
corr = df_corr[numeric_cols].corr()

fig = px.imshow(corr, text_auto=True, title='Correlation Heatmap')
fig.show()

In [ ]:
# --- Save cleaned dataset for next notebook/train.py ---
df.to_csv(r"C:\Users\Sruthi K\OneDrive\Desktop\Customer-Churn-Analytics\data\telecom_churn.csv", index=False)
print("Cleaned dataset saved ✅")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Load cleaned data
df = pd.read_csv(r"C:\Users\Sruthi K\OneDrive\Desktop\Customer-Churn-Analytics\data\telecom_churn.csv")
df.head()

In [ ]:
# --- Encode target variable ---
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

In [ ]:
# --- Identify column types ---
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'SeniorCitizen']
multi_cat_cols = ['gender', 'MultipleLines', 'InternetService', 'OnlineSecurity',
                   'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
                   'StreamingMovies', 'Contract', 'PaymentMethod']
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

print("Binary:", binary_cols)
print("Multi-category:", multi_cat_cols)
print("Numeric:", numeric_cols)

In [ ]:
# --- Encode binary columns (Yes/No -> 1/0), safely ---
for col in binary_cols:
    print(col, "unique values before:", df[col].unique())
    df[col] = df[col].map({'Yes': 1, 'No': 0})
    print(col, "unique values after:", df[col].unique())
    print("---")

In [ ]:
# --- One-hot encode multi-category columns ---
df_encoded = pd.get_dummies(df, columns=multi_cat_cols, drop_first=True)
print("Shape after encoding:", df_encoded.shape)
df_encoded.head()

In [ ]:
# --- Separate features and target ---
X = df_encoded.drop(['Churn'], axis=1)
y = df_encoded['Churn']

print("X shape:", X.shape)
print("y distribution:\n", y.value_counts(normalize=True))

In [ ]:
# --- Train/test split ---
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

In [ ]:
# --- Scale numeric features (fit on train only, apply to both) ---
scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

X_train[numeric_cols].describe()

In [ ]:
# --- Save processed data + scaler for train.py later ---
import joblib

X_train.to_csv(r"C:\Users\Sruthi K\OneDrive\Desktop\Customer-Churn-Analytics\data\X_train.csv", index=False)
X_test.to_csv(r"C:\Users\Sruthi K\OneDrive\Desktop\Customer-Churn-Analytics\data\X_test.csv", index=False)
y_train.to_csv(r"C:\Users\Sruthi K\OneDrive\Desktop\Customer-Churn-Analytics\data\y_train.csv", index=False)
y_test.to_csv(r"C:\Users\Sruthi K\OneDrive\Desktop\Customer-Churn-Analytics\data\y_test.csv", index=False)

joblib.dump(scaler, r"C:\Users\Sruthi K\OneDrive\Desktop\Customer-Churn-Analytics\model\scaler.pkl")
print("Saved train/test splits and scaler ✅")

In [ ]:
import os
print(os.listdir(r"C:\Users\Sruthi K\OneDrive\Desktop\Customer-Churn-Analytics"))

In [ ]:
# Check for any remaining non-numeric columns
non_numeric = X_train.select_dtypes(include='object').columns.tolist()
print("Non-numeric columns still in X_train:", non_numeric)